### **Imports**

In [ ]:
import os
from pathlib import Path
import pandas as pd
from IPython.display import display

if not (Path.cwd() / "src").is_dir() and (Path.cwd().parent / "src").is_dir():
    os.chdir(Path.cwd().parent)

%load_ext autoreload
%autoreload 2

from src.config import WORKING_DIR
from src.utils.visualization import (
    load_video_ablation_reports,
    plot_val_tracker_calibration,
    plot_video_ablation_dashboard,
)

### **Adaptive ROI & MOT Video Ablation Analysis (OpenVINO INT8)**

In [ ]:
ABLATION_DIR = WORKING_DIR / "video_ablation"
REPORTS_DIR = ABLATION_DIR / "reports"
reports = load_video_ablation_reports(ABLATION_DIR, eval_split="test")
results_df = reports["results_df"]
metadata = reports["metadata"]
val_grid_df = reports["val_grid_df"]
val_roi_df = reports["val_roi_df"]
pd.set_option("display.max_columns", None)

### **Calibrated Val Hyperparameters & Environment Provenance**

In [ ]:
provenance_df = pd.DataFrame([
    {"Item": "Eval Split", "Value": metadata.get("eval_split")},
    {"Item": "Sequences Evaluated", "Value": metadata.get("num_sequences")},
    {"Item": "Model 1280 (Search)", "Value": metadata.get("model_1280")},
    {"Item": "Model 640 (Track)", "Value": metadata.get("model_640")},
    {"Item": "Best ByteTrack Config (Val)", "Value": str((metadata.get("tracker_configs") or {}).get("bytetrack"))},
    {"Item": "Best OC-SORT Config (Val)", "Value": str((metadata.get("tracker_configs") or {}).get("ocsort"))},
    {"Item": "Calibrated max_lost_frames (Val)", "Value": metadata.get("max_lost_frames")},
    {"Item": "det_conf / seed_conf", "Value": f"{metadata.get('det_conf')} / {metadata.get('seed_conf')}"},
])
display(provenance_df.style.hide(axis="index"))

### **Validation Split: ByteTrack vs OC-SORT Calibration & ROI Hysteresis**

In [ ]:
if not val_grid_df.empty:
    best_val_rows = (
        val_grid_df.sort_values(["HOTA", "IDF1"], ascending=[False, False])
        .groupby("tracker", as_index=False)
        .first()
    )
    display(best_val_rows.style.format(precision=3, na_rep="-").hide(axis="index"))

if not val_roi_df.empty:
    display(val_roi_df.style.format(precision=3, na_rep="-").hide(axis="index"))

if not val_grid_df.empty:
    plot_val_tracker_calibration(
        val_grid_df,
        val_roi_df=val_roi_df if not val_roi_df.empty else None,
        save_path=REPORTS_DIR / "val_tracker_calibration.png",
    )

### **Test Split: Full 1280 vs Resize 640 vs ROI Static vs ROI Kalman**

In [ ]:
view_df = results_df.copy()
full_baselines = view_df[view_df["mode"] == "full_1280"].set_index("tracker")["e2e_mean_ms"].to_dict()
view_df["speedup_vs_full1280"] = view_df.apply(
    lambda r: (full_baselines[r["tracker"]] / r["e2e_mean_ms"]) if r["tracker"] in full_baselines and r["e2e_mean_ms"] > 0 else float("nan"),
    axis=1,
)
view_df["track_recall_small (%)"] = view_df["track_recall_small"] * 100.0
display_cols = [
    "mode", "tracker", "e2e_fps", "e2e_mean_ms", "e2e_p95_ms", "speedup_vs_full1280",
    "pct_track_frames", "search_transitions", "HOTA", "IDF1", "MOTA", "DetA", "AssA",
    "IDSW", "Frag", "track_recall_small (%)",
]
display(view_df[[c for c in display_cols if c in view_df.columns]].style.format(precision=2, na_rep="-").hide(axis="index"))

plot_video_ablation_dashboard(
    results_df,
    target_fps=float(metadata.get("fps", 30)),
    save_path=REPORTS_DIR / "test_video_ablation_dashboard.png",
)